# Data Preparation & Chunking for LLM Retrieval

This notebook prepares customer feedback and related metadata for use in a
retrieval-augmented generation (RAG) system. The focus is on structuring and
chunking data in a way that enables accurate, grounded LLM responses.

In [1]:
import pandas as pd
import numpy as np
import re

## Load Customer Feedback Data

We reuse cleaned customer feedback from earlier NLP work and enrich it with
metadata that will later help the LLM reason about customer issues.

In [2]:
df = pd.read_csv("../data/raw/reviews_sample.csv")

df = df[["Text", "Score"]].dropna()

df.head()

,Text,Score
0,Having tried a couple of other brands of glute...,5
1,My cat loves these treats. If ever I can't fin...,5
2,A little less than I expected. It tends to ha...,3
3,"First there was Frosted Mini-Wheats, in origin...",2
4,and I want to congratulate the graphic artist ...,5


## Create Structured Metadata

LLMs perform better when unstructured text is accompanied by structured context.
We derive sentiment labels and attach them as metadata for retrieval.

In [3]:
def map_sentiment(score):
    if score >= 4:
        return "positive"
    elif score <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["Score"].apply(map_sentiment)

df["sentiment"].value_counts()

sentiment
positive    7852
negative    1398
neutral      750
Name: count, dtype: int64

## Normalize Text for Chunking

We lightly normalize text to reduce noise while preserving meaning.
Over-cleaning can remove useful semantic signals.

In [4]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["Text"].astype(str).apply(normalize_text)

df[["clean_text", "sentiment"]].head()

,clean_text,sentiment
0,having tried a couple of other brands of glute...,positive
1,my cat loves these treats. if ever i can't fin...,positive
2,a little less than i expected. it tends to hav...,neutral
3,"first there was frosted mini-wheats, in origin...",negative
4,and i want to congratulate the graphic artist ...,positive


## Why Chunking Matters

LLMs have context length limits. Large documents must be broken into smaller
chunks to ensure relevant information can be retrieved and used correctly.
Poor chunking leads to hallucinations and missed evidence.

In [5]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.split()) > 30:
            chunks.append(chunk)

    return chunks

## Apply Chunking to Customer Feedback

Each review is split into overlapping chunks to preserve local context.
Metadata is attached to every chunk.

In [6]:
records = []

for _, row in df.iterrows():
    chunks = chunk_text(row["clean_text"])
    for chunk in chunks:
        records.append({
            "text_chunk": chunk,
            "sentiment": row["sentiment"]
        })

chunked_df = pd.DataFrame(records)

chunked_df.head(), chunked_df.shape

(                                          text_chunk sentiment
 0  having tried a couple of other brands of glute...  positive
 1  my cat loves these treats. if ever i can't fin...  positive
 2  first there was frosted mini-wheats, in origin...  negative
 3  fiber, 12g of sugar, 5g of protein and 200mg o...  negative
 4  and i want to congratulate the graphic artist ...  positive,
 (9010, 2))

## Inspect Chunk Quality

We manually inspect chunks to ensure they are coherent, meaningful,
and not overly fragmented.

In [7]:
chunked_df.sample(5, random_state=42)

,text_chunk,sentiment
6976,kettle chips are thicker and crunchier (though...,positive
6770,was recommended this product after gastric byp...,positive
7506,i have a cat with a heart condition that takes...,positive
3131,after reading the positive reviews about the v...,positive
932,this was a split in the family - my husband an...,neutral


## Save Processed Chunked Data

The chunked dataset will be used for embedding generation and vector search
in the retrieval layer.

In [8]:
chunked_df.to_csv("../data/processed/review_chunks.csv", index=False)

## Key Takeaways

- Chunking strategy directly affects retrieval quality in LLM systems.
- Attaching sentiment metadata enables filtered and targeted retrieval.
- Light text normalization preserves semantic meaning while reducing noise.